#Clos.py


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from torch.optim import AdamW, LBFGS
from torch import Tensor

def make_fourier_eye(
    hidden_dim: int = 768,
    seq_len: int = 512,
    device: str = "cuda",
    base: float = 10000.0,
    include_sin_cos: bool = True
) -> torch.Tensor:
    pos = torch.arange(seq_len, device=device).float()
    inv_freq = 1.0 / (base ** (torch.arange(0, hidden_dim, 2, device=device).float() / hidden_dim))
    inv_freq = inv_freq[:hidden_dim // 2]
    angles = pos[:, None] * inv_freq[None, :]
    if include_sin_cos:
        fourier = torch.cat([angles.cos(), angles.sin()], dim=-1)
    else:
        fourier = angles.sin()
    return fourier.contiguous()


class Clos(nn.Module):
    def __init__(self, in_features=768, out_features=None, channel=3, switches=None, bias=True, middle_switch_multiplier=4):
        super(Clos, self).__init__()
        self.in_features = in_features
        self.out_features = out_features if out_features is not None else in_features
        self.channel = channel
        self.bias = bias
        self.middle_switch_multiplier = middle_switch_multiplier
        self.switches = {}

        self.find_factors()
        if switches is not None:
            self.switches.update(switches)

        k = 1.0 / math.sqrt(in_features)
        self.weight1 = nn.Parameter(torch.Tensor(self.switches['bin'], self.switches['b1'], self.switches['b2']))
        self.weight2 = nn.Parameter(torch.Tensor(self.switches['b1'], self.switches['b2'], self.switches['b3']))
        self.weight3 = nn.Parameter(torch.Tensor(self.switches['b2'], self.switches['b3'], self.switches['bout']))

        if self.bias:
            self.bias1 = nn.Parameter(torch.Tensor(self.switches['b1']))
            self.bias2 = nn.Parameter(torch.Tensor(self.switches['b2']))
            self.bias3 = nn.Parameter(torch.Tensor(self.switches['b3']))
        else:
            self.register_parameter('bias1', None)
            self.register_parameter('bias2', None)
            self.register_parameter('bias3', None)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.weight1, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.weight2, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.weight3, a=math.sqrt(5))
        if self.bias:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight1)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias1, -bound, bound)
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight2)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias2, -bound, bound)
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight3)
            bound = 1 / math.sqrt(fan_in) if fan_in > 0 else 0
            nn.init.uniform_(self.bias3, -bound, bound)

    def find_factors(self):
        for i in range(int(math.sqrt(self.in_features)), 0, -1):
            if self.in_features % i == 0:
                self.switches['bin'] = i
                self.switches['b1'] = self.in_features // i
                break
        for i in range(int(math.sqrt(self.out_features)), 0, -1):
            if self.out_features % i == 0:
                self.switches['bout'] = i
                self.switches['b3'] = self.out_features // i
                break
        self.switches['b2'] = self.middle_switch_multiplier * self.switches['bin']

    def __repr__(self):
        return (f"Clos(in={self.in_features}, out={self.out_features}, "
                f"bias={self.bias}, bin={self.switches['bin']}, b1={self.switches['b1']}, "
                f"b2={self.switches['b2']}, b3={self.switches['b3']}, bout={self.switches['bout']}, "
                f"channel={self.channel})")

    def channel2(self, x: torch.Tensor) -> torch.Tensor:
        b = x.shape[0]
        x = x.view(b, self.switches['bin'], self.switches['b1'])
        if self.bias:
            x = torch.einsum('bnr,nrm->bmr', x, self.weight1) + self.bias1
            x = torch.einsum('bmr,rmn->bnm', x, self.weight2) + self.bias2
            x = torch.einsum('bnm,mro->bor', x, self.weight3) + self.bias3
        else:
            x = torch.einsum('bnr,nrm->bmr', x, self.weight1)
            x = torch.einsum('bmr,rmn->bnm', x, self.weight2)
            x = torch.einsum('bnm,mro->bor', x, self.weight3)
        return x.reshape(b, -1)

    def channel3(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _ = x.shape
        x = x.view(b, c, self.switches['bin'], self.switches['b1'])
        if self.bias:
            x = torch.einsum('bcnr,nrm->bcmr', x, self.weight1) + self.bias1
            x = torch.einsum('bcmr,rmn->bcnm', x, self.weight2) + self.bias2
            x = torch.einsum('bcnm,mro->bcor', x, self.weight3) + self.bias3
        else:
            x = torch.einsum('bcnr,nrm->bcmr', x, self.weight1)
            x = torch.einsum('bcmr,rmn->bcnm', x, self.weight2)
            x = torch.einsum('bcnm,mro->bcor', x, self.weight3)
        return x.reshape(b, c, -1)

    def forward(self, input: Tensor) -> Tensor:
        if self.channel == 2:
            return self.channel2(input)
        elif self.channel == 3:
            return self.channel3(input)


def transfer_fc_to_clos(fc: nn.Linear, channel: int = 2, max_steps: int = 2500, W_lr: float = 1e-3, B_lr: float = 1e-5, verbose: bool = False):
    device = fc.weight.device
    in_f, out_f = fc.in_features, fc.out_features
    clos = Clos(in_features=in_f, out_features=out_f, channel=channel, bias=True).to(device)
    clos.train()
    target = fc.weight.T.contiguous()

    if channel == 2:
        eye = torch.eye(in_f, device=device)
    else:
        eye = make_fourier_eye(hidden_dim=in_f, seq_len=in_f, device=device).unsqueeze(0)

    optimizer = AdamW(clos.parameters(), lr=W_lr, weight_decay=1e-3)
    for step in range(max_steps):
        optimizer.zero_grad()
        pred = clos(eye)
        loss = F.mse_loss(pred, target)
        loss.backward()
        optimizer.step()
        if verbose and (step % 500 == 0 or step == max_steps-1):
            print(f"step {step:4d} | loss {loss.item():.8f}")

    print("LBFGS polish...")
    optimizer = LBFGS(clos.parameters(), lr=0.1, max_iter=800, line_search_fn="strong_wolfe")
    def closure():
        optimizer.zero_grad()
        pred = clos(eye)
        loss = F.mse_loss(pred, target)
        loss.backward()
        return loss
    for _ in range(8):
        optimizer.step(closure)

    if fc.bias is not None:
        zero_input = torch.zeros(1, in_f, device=device)
        target_bias = fc.bias.clone().unsqueeze(0).expand(out_f, out_f)
        bias_params = [clos.bias1, clos.bias2, clos.bias3]
        bias_opt = AdamW(bias_params, lr=B_lr, weight_decay=1e-2)
        for _ in range(1200):
            bias_opt.zero_grad()
            out = clos(zero_input)
            b_loss = F.mse_loss(out, target_bias)
            b_loss.backward()
            bias_opt.step()
    return clos

#Train_mnist.py

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

class MNIST_Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 784)
        self.fc2 = nn.Linear(784, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()
        nn.init.kaiming_normal_(self.fc1.weight, mode='fan_in', nonlinearity='relu')
        nn.init.kaiming_normal_(self.fc2.weight, mode='fan_in', nonlinearity='relu')
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


def train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device):
    model.train()
    total_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
            loss = criterion(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    return total_loss / len(train_loader.dataset), 100. * correct / len(train_loader.dataset)


@torch.no_grad()
def test(model, test_loader, use_amp, device):
    model.eval()
    correct = 0
    for data, target in test_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    return 100. * correct / len(test_loader.dataset)


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)

    model = MNIST_Net().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=0.01, steps_per_epoch=len(train_loader), epochs=15, pct_start=0.3, anneal_strategy='cos'
    )
    scaler = torch.amp.GradScaler(enabled=use_amp)

    print("Training started...")
    for epoch in range(1, 16):
        train_loss, train_acc = train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device)
        test_acc = test(model, test_loader, use_amp, device)
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | Train: {train_acc:.2f}% | Test: {test_acc:.3f}%")

    torch.save(model.state_dict(), "mnist_784_256_10.pth")
    print("Model saved to mnist_784_256_10.pth")


if __name__ == "__main__":
    main()

Training started...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch  1 | Loss: 0.4509 | Train: 87.59% | Test: 95.060%
Epoch  2 | Loss: 0.1301 | Train: 96.06% | Test: 95.680%
Epoch  3 | Loss: 0.0947 | Train: 97.02% | Test: 96.840%
Epoch  4 | Loss: 0.0870 | Train: 97.33% | Test: 96.480%
Epoch  5 | Loss: 0.0680 | Train: 97.96% | Test: 97.060%
Epoch  6 | Loss: 0.0579 | Train: 98.21% | Test: 97.520%
Epoch  7 | Loss: 0.0354 | Train: 98.88% | Test: 97.950%
Epoch  8 | Loss: 0.0271 | Train: 99.17% | Test: 97.540%
Epoch  9 | Loss: 0.0176 | Train: 99.42% | Test: 98.210%
Epoch 10 | Loss: 0.0079 | Train: 99.75% | Test: 98.390%
Epoch 11 | Loss: 0.0020 | Train: 99.95% | Test: 98.490%
Epoch 12 | Loss: 0.0006 | Train: 99.99% | Test: 98.540%
Epoch 13 | Loss: 0.0003 | Train: 100.00% | Test: 98.560%
Epoch 14 | Loss: 0.0002 | Train: 100.00% | Test: 98.560%
Epoch 15 | Loss: 0.0002 | Train: 100.00% | Test: 98.560%
Model saved to mnist_784_256_10.pth


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW, LBFGS
from clos1 import Clos

def transfer_linear_to_clos_identity(
    linear,
    n_components: int = 512,
    channel: int = 2,
    max_steps: int = 12000,
    W_lr: float = 3e-3,
    B_lr: float = 8e-4,
    verbose: bool = True,
    device = None
) -> Clos:

    if device is None:
        device = linear.weight.device if hasattr(linear, 'weight') else torch.device("cuda" if torch.cuda.is_available() else "cpu")

    n = n_components

    clos = Clos(
        in_features    = n,
        out_features   = n,
        channel        = channel,
        bias           = True,
        middle_switch_multiplier = 4
    ).to(device)

    eye_input = torch.eye(n, device=device)
    target    = torch.eye(n, device=device)

    print(f"eye_input shape: {eye_input.shape}")
    print(f"target shape:    {target.shape}")
    print(f"Clos switches:   {clos.switches}")

    clos.train()
    optimizer = AdamW(clos.parameters(), lr=W_lr, weight_decay=1e-3)

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = clos(eye_input)
        loss = F.mse_loss(pred, target)
        loss.backward()
        optimizer.step()

        if verbose and (step % 1500 == 0 or step == max_steps - 1):
            print(f"[{step:5d}] MSE = {loss.item():.8f}")

    print("\nLBFGS polish phase...")
    optimizer_lbfgs = LBFGS(
        clos.parameters(),
        lr=0.08,
        max_iter=600,
        line_search_fn="strong_wolfe"
    )

    def closure():
        optimizer_lbfgs.zero_grad()
        pred = clos(eye_input)
        loss = F.mse_loss(pred, target)
        loss.backward()
        return loss

    optimizer_lbfgs.step(closure)

    final_loss = F.mse_loss(clos(eye_input), target).item()
    print(f"Final MSE after LBFGS: {final_loss:.8f}")

    print("\nBias fine-tuning...")
    zero_input = torch.zeros(1, n, device=device)
    target_zero = torch.zeros(1, n, device=device)

    bias_opt = AdamW([clos.bias1, clos.bias2, clos.bias3], lr=B_lr, weight_decay=1e-3)

    for it in range(1200):
        bias_opt.zero_grad()
        out = clos(zero_input)
        b_loss = F.mse_loss(out, target_zero)
        b_loss.backward()
        bias_opt.step()

        if verbose and (it % 200 == 0 or it == 1199):
            print(f"  bias step {it:4d} | loss {b_loss.item():.2e}")

    print(f"Final bias loss: {b_loss.item():.2e}")

    save_path = f"clos_identity_{n}.pth"
    torch.save(clos.state_dict(), save_path)
    print(f"Model saved to: {save_path}")

    return clos


if __name__ == "__main__":
    from train_mni import MNIST_Net

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = MNIST_Net().to(device)
    model.load_state_dict(torch.load("mnist_784_256_10.pth", map_location=device))


    clos = transfer_linear_to_clos_identity(
        linear       = model.fc1,
        n_components = 512,
        channel      = 2,
        max_steps    = 12000,
        W_lr         = 3e-3,
        verbose      = True
    )

Using device: cuda
eye_input shape: torch.Size([512, 512])
target shape:    torch.Size([512, 512])
Clos switches:   {'bin': 16, 'b1': 32, 'bout': 16, 'b3': 32, 'b2': 64}
[    0] MSE = 0.00918093
[ 1500] MSE = 0.00170517
[ 3000] MSE = 0.00170517
[ 4500] MSE = 0.00170517
[ 6000] MSE = 0.00170537
[ 7500] MSE = 0.00170525
[ 9000] MSE = 0.00174642
[10500] MSE = 0.00170530
[11999] MSE = 0.00170522

LBFGS polish phase...
Final MSE after LBFGS: 0.00170522

Bias fine-tuning...
  bias step    0 | loss 2.00e-04
  bias step  200 | loss 3.33e-13
  bias step  400 | loss 7.99e-14
  bias step  600 | loss 7.99e-14
  bias step  800 | loss 8.00e-14
  bias step 1000 | loss 7.97e-14
  bias step 1199 | loss 8.07e-14
Final bias loss: 8.07e-14
Model saved to: clos_identity_512.pth


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW, LBFGS
from sklearn.decomposition import PCA
import numpy as np
from clos1 import Clos

def transfer_fc1_to_clos_projected(
    linear: nn.Linear,
    n_components: int = 512,
    channel: int = 2,
    max_steps: int = 12000,
    W_lr: float = 3e-3,
    B_lr: float = 8e-4,
    verbose: bool = True,
    device = None
) -> Clos:

    if device is None:
        device = linear.weight.device

    in_dim = linear.in_features
    orig_out = linear.out_features

    W = linear.weight.data.cpu().numpy()
    pca = PCA(n_components=n_components)
    pca.fit(W)

    proj_basis = torch.from_numpy(pca.components_).float().to(device)

    explained = sum(pca.explained_variance_ratio_)
    print(f"Explained variance ({n_components}): {explained:.5f}")


    clos = Clos(
        in_features=in_dim,
        out_features=n_components,
        channel=channel,
        bias=True,
        middle_switch_multiplier=4
    ).to(device)

    eye = torch.eye(in_dim, device=device)

    with torch.no_grad():
        original_out = linear(eye)
        target = original_out @ proj_basis.t()

    eye_input = eye
    target    = target

    print(f"eye_input shape: {eye_input.shape}")
    print(f"target shape:    {target.shape}")

    clos.train()
    optimizer = AdamW(clos.parameters(), lr=W_lr, weight_decay=1e-3)

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = clos(eye_input)
        loss = F.mse_loss(pred, target)
        loss.backward()
        optimizer.step()
        ...

        if verbose and (step % 1500 == 0 or step == max_steps - 1):
            print(f"[{step:5d}] MSE = {loss.item():.8f}")

    print("\nLBFGS polish...")
    optimizer_lbfgs = LBFGS(
        clos.parameters(),
        lr=0.08,
        max_iter=600,
        line_search_fn="strong_wolfe"
    )

    def closure():
        optimizer_lbfgs.zero_grad()
        pred = clos(eye_input)
        loss = F.mse_loss(pred, target)
        loss.backward()
        return loss

    optimizer_lbfgs.step(closure)

    final_mse = F.mse_loss(clos(eye_input), target).item()
    print(f"Final MSE after LBFGS: {final_mse:.8f}")

    print("\nBias fine-tuning...")
    zero_input = torch.zeros(1, in_dim, device=device)
    target_zero = torch.zeros(1, 1, n_components, device=device)

    bias_opt = AdamW([clos.bias1, clos.bias2, clos.bias3], lr=B_lr, weight_decay=1e-3)

    for it in range(1200):
        bias_opt.zero_grad()
        out = clos(zero_input)
        out = out.view(1, -1, n_components) if out.dim() == 2 else out
        b_loss = F.mse_loss(out, target_zero.expand_as(out))
        b_loss.backward()
        bias_opt.step()

        if verbose and (it % 200 == 0 or it == 1199):
            print(f"  bias step {it:4d} | loss {b_loss.item():.2e}")

    print(f"Final bias loss: {b_loss.item():.2e}")

    save_path = f"clos_projected_784_to_{n_components}.pth"
    torch.save(clos.state_dict(), save_path)
    print(f"Хадгаллаа: {save_path}")

    return clos


if __name__ == "__main__":
    from train_mni import MNIST_Net

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    model = MNIST_Net().to(device)
    model.load_state_dict(torch.load("mnist_784_256_10.pth", map_location=device))

    clos = transfer_fc1_to_clos_projected(
        linear=model.fc1,
        n_components=512,
        channel=2,
        max_steps=12000,
        W_lr=3e-3,
        B_lr=8e-4,
        verbose=True
    )

Device: cuda
Explained variance (512): 0.98981
eye_input shape: torch.Size([784, 784])
target shape:    torch.Size([784, 512])
[    0] MSE = 0.02678171
[ 1500] MSE = 0.00289604
[ 3000] MSE = 0.00289612
[ 4500] MSE = 0.00289687
[ 6000] MSE = 0.00289709
[ 7500] MSE = 0.00291826
[ 9000] MSE = 0.00290028
[10500] MSE = 0.00289645
[11999] MSE = 0.00289620

LBFGS polish...
Final MSE after LBFGS: 0.00289614

Bias fine-tuning...
  bias step    0 | loss 3.52e-03
  bias step  200 | loss 1.02e-11
  bias step  400 | loss 1.20e-17
  bias step  600 | loss 1.07e-17
  bias step  800 | loss 1.03e-17
  bias step 1000 | loss 1.34e-17
  bias step 1199 | loss 1.30e-17
Final bias loss: 1.30e-17
Хадгаллаа: clos_projected_784_to_512.pth


In [ ]:
transform = transforms.Compose([transforms.ToTensor()])
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


In [ ]:
import torch
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from train_mni import MNIST_Net, test

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = MNIST_Net().to(device)
model.load_state_dict(torch.load("mnist_784_256_10.pth", map_location=device))

acc_original = test(model, test_loader, use_amp=True, device=device)
print(f"Original model accuracy: {acc_original:.3f}%")

clos = Clos(
    in_features=784,
    out_features=512,
    channel=2
).to(device)

clos.load_state_dict(torch.load("clos_projected_784_to_512.pth", map_location=device))

model.fc1 = clos
model.fc2 = nn.Linear(512, 256).to(device)

print("Model updated: fc1 = Clos(784→512), fc2 = Linear(512→256)")


acc_clos = test(model, test_loader, use_amp=True, device=device)
print(f"Clos (784→512) accuracy: {acc_clos:.3f}%")
print(f"Параметрын тоо (clos): {sum(p.numel() for p in clos.parameters()):,}")

Device: cuda
Original model accuracy: 98.560%
Model updated: fc1 = Clos(784→512), fc2 = Linear(512→256)
Clos (784→512) accuracy: 10.120%
Параметрын тоо (clos): 245,676


#1

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

class MNIST_Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 784)
        self.fc2 = nn.Linear(784, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()

        # Proper Kaiming initialization for ReLU layers
        nn.init.kaiming_normal_(self.fc1.weight, mode='fan_in', nonlinearity='relu')
        nn.init.kaiming_normal_(self.fc2.weight, mode='fan_in', nonlinearity='relu')
        # Last layer: default Xavier or zero bias is fine, but many use normal init
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)          # (B,1,28,28) -> (B,784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)                    # No activation on final layer for CE Loss
        return x


def train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device):
    model.train()
    total_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
            loss = criterion(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # OneCycleLR should be stepped every batch
        scheduler.step()
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    avg_loss = total_loss / len(train_loader.dataset)
    acc = 100. * correct / len(train_loader.dataset)
    return avg_loss, acc

@torch.no_grad()
def test(model, test_loader, use_amp, device):
    model.eval()
    correct = 0
    for data, target in test_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    acc = 100. * correct / len(test_loader.dataset)
    return acc

def main():
    Epochs = 15
    # Device & Mixed Precision
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    use_amp = device.type == "cuda"  # Automatic Mixed Precision only on GPU
    # Data
    transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True,  num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)
    model = MNIST_Net().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.01,
        steps_per_epoch=len(train_loader),
        epochs=Epochs,
        pct_start=0.3,
        anneal_strategy='cos'
    )
    scaler = torch.amp.GradScaler(enabled=use_amp)
    print("\nTraining started...\n")
    start_time = time.time()
    for epoch in range(1, Epochs+1):
        train_loss, train_acc = train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device)
        test_acc = test(model, test_loader, use_amp, device)
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:6.2f}% | Test Acc: {test_acc:6.3f}%")
    total_time = time.time() - start_time
    print(f"\nFinished! Total time: {total_time:.1f} sec")
    print(f"Test accuracy: {test_acc:.3f}%")
    torch.save(model.state_dict(), "mnist_784_256_10.pth")
    print("Model saved → mnist_784_256_10.pth")


if __name__ == "__main__":
    main()

Device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 491kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.55MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.2MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



Training started...

Epoch  1 | Loss: 0.4386 | Train Acc:  87.91% | Test Acc: 95.380%
Epoch  2 | Loss: 0.1287 | Train Acc:  96.16% | Test Acc: 95.440%
Epoch  3 | Loss: 0.0975 | Train Acc:  96.99% | Test Acc: 93.820%
Epoch  4 | Loss: 0.0873 | Train Acc:  97.30% | Test Acc: 96.750%
Epoch  5 | Loss: 0.0695 | Train Acc:  97.75% | Test Acc: 97.030%
Epoch  6 | Loss: 0.0523 | Train Acc:  98.36% | Test Acc: 97.320%
Epoch  7 | Loss: 0.0439 | Train Acc:  98.60% | Test Acc: 97.460%
Epoch  8 | Loss: 0.0254 | Train Acc:  99.15% | Test Acc: 97.780%
Epoch  9 | Loss: 0.0204 | Train Acc:  99.37% | Test Acc: 98.260%
Epoch 10 | Loss: 0.0075 | Train Acc:  99.75% | Test Acc: 98.410%
Epoch 11 | Loss: 0.0028 | Train Acc:  99.92% | Test Acc: 98.490%
Epoch 12 | Loss: 0.0006 | Train Acc:  99.99% | Test Acc: 98.580%
Epoch 13 | Loss: 0.0003 | Train Acc: 100.00% | Test Acc: 98.570%
Epoch 14 | Loss: 0.0002 | Train Acc: 100.00% | Test Acc: 98.570%
Epoch 15 | Loss: 0.0002 | Train Acc: 100.00% | Test Acc: 98.570%

Fi

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

class MNIST_Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 784)
        self.fc2 = nn.Linear(784, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()

        nn.init.kaiming_normal_(self.fc1.weight, mode='fan_in', nonlinearity='relu')
        nn.init.kaiming_normal_(self.fc2.weight, mode='fan_in', nonlinearity='relu')
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


def train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device):
    model.train()
    total_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
            loss = criterion(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    avg_loss = total_loss / len(train_loader.dataset)
    acc = 100. * correct / len(train_loader.dataset)
    return avg_loss, acc


@torch.no_grad()
def test(model, test_loader, use_amp, device):
    model.eval()
    correct = 0
    for data, target in test_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    acc = 100. * correct / len(test_loader.dataset)
    return acc


def main():
    Epochs = 15
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    use_amp = device.type == "cuda"

    transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True,  num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)

    model = MNIST_Net().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.01,
        steps_per_epoch=len(train_loader),
        epochs=Epochs,
        pct_start=0.3,
        anneal_strategy='cos'
    )
    scaler = torch.amp.GradScaler(enabled=use_amp)

    print("\nTraining started...\n")
    start_time = time.time()
    for epoch in range(1, Epochs+1):
        train_loss, train_acc = train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device)
        test_acc = test(model, test_loader, use_amp, device)
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:6.2f}% | Test Acc: {test_acc:6.3f}%")
    total_time = time.time() - start_time
    print(f"\nFinished! Total time: {total_time:.1f} sec")
    print(f"Test accuracy: {test_acc:.3f}%")
    torch.save(model.state_dict(), "mnist_784.pth")
    print("Model saved → mnist_784.pth")


if __name__ == "__main__":
    main()

Device: cuda

Training started...



/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch  1 | Loss: 0.4327 | Train Acc:  87.98% | Test Acc: 95.010%
Epoch  2 | Loss: 0.1309 | Train Acc:  96.01% | Test Acc: 96.780%
Epoch  3 | Loss: 0.1000 | Train Acc:  96.90% | Test Acc: 94.870%
Epoch  4 | Loss: 0.0928 | Train Acc:  97.14% | Test Acc: 96.100%
Epoch  5 | Loss: 0.0725 | Train Acc:  97.72% | Test Acc: 97.460%
Epoch  6 | Loss: 0.0502 | Train Acc:  98.41% | Test Acc: 97.660%
Epoch  7 | Loss: 0.0370 | Train Acc:  98.78% | Test Acc: 97.920%
Epoch  8 | Loss: 0.0259 | Train Acc:  99.16% | Test Acc: 97.960%
Epoch  9 | Loss: 0.0172 | Train Acc:  99.45% | Test Acc: 98.280%
Epoch 10 | Loss: 0.0093 | Train Acc:  99.70% | Test Acc: 98.390%
Epoch 11 | Loss: 0.0044 | Train Acc:  99.86% | Test Acc: 98.440%
Epoch 12 | Loss: 0.0011 | Train Acc:  99.97% | Test Acc: 98.550%
Epoch 13 | Loss: 0.0003 | Train Acc: 100.00% | Test Acc: 98.610%
Epoch 14 | Loss: 0.0002 | Train Acc: 100.00% | Test Acc: 98.570%
Epoch 15 | Loss: 0.0002 | Train Acc: 100.00% | Test Acc: 98.580%

Finished! Total time: 11

#MNIST DIM

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

class MNIST_Net_512(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.relu = nn.ReLU()

        # Proper Kaiming initialization for ReLU layers
        nn.init.kaiming_normal_(self.fc1.weight, mode='fan_in', nonlinearity='relu')
        nn.init.kaiming_normal_(self.fc2.weight, mode='fan_in', nonlinearity='relu')
        # Last layer: default Xavier or zero bias is fine, but many use normal init
        nn.init.xavier_normal_(self.fc3.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.zeros_(self.fc2.bias)
        nn.init.zeros_(self.fc3.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)          # (B,1,28,28) -> (B,784)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)                    # No activation on final layer for CE Loss
        return x

def train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device):
    model.train()
    total_loss = 0.0
    correct = 0
    for data, target in train_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
            loss = criterion(output, target)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        # OneCycleLR should be stepped every batch
        scheduler.step()
        total_loss += loss.item() * data.size(0)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    avg_loss = total_loss / len(train_loader.dataset)
    acc = 100. * correct / len(train_loader.dataset)
    return avg_loss, acc

@torch.no_grad()
def test(model, test_loader, use_amp, device):
    model.eval()
    correct = 0
    for data, target in test_loader:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=use_amp):
            output = model(data)
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
    acc = 100. * correct / len(test_loader.dataset)
    return acc

def main():
    Epochs = 15
    # Device & Mixed Precision
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    use_amp = device.type == "cuda"  # Automatic Mixed Precision only on GPU
    # Data
    transform = transforms.Compose([transforms.ToTensor()])
    train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    test_dataset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True,  num_workers=4, pin_memory=True)
    test_loader  = DataLoader(test_dataset,  batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)
    model = MNIST_Net().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=0.01,
        steps_per_epoch=len(train_loader),
        epochs=Epochs,
        pct_start=0.3,
        anneal_strategy='cos'
    )
    scaler = torch.amp.GradScaler(enabled=use_amp)
    print("\nTraining started...\n")
    start_time = time.time()
    for epoch in range(1, Epochs+1):
        train_loss, train_acc = train_epoch(model, optimizer, criterion, train_loader, scaler, scheduler, use_amp, device)
        test_acc = test(model, test_loader, use_amp, device)
        print(f"Epoch {epoch:2d} | Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:6.2f}% | Test Acc: {test_acc:6.3f}%")
    total_time = time.time() - start_time
    print(f"\nFinished! Total time: {total_time:.1f} sec")
    print(f"Test accuracy: {test_acc:.3f}%")
    torch.save(model.state_dict(), "mnist_512_256_10.pth")
    print("Model saved → mnist_512_256_10.pth")


if __name__ == "__main__":
    main()

Device: cuda

Training started...

Epoch  1 | Loss: 0.4502 | Train Acc:  87.59% | Test Acc: 95.230%
Epoch  2 | Loss: 0.1273 | Train Acc:  96.14% | Test Acc: 95.850%
Epoch  3 | Loss: 0.0999 | Train Acc:  96.92% | Test Acc: 95.900%
Epoch  4 | Loss: 0.0894 | Train Acc:  97.20% | Test Acc: 96.820%
Epoch  5 | Loss: 0.0711 | Train Acc:  97.82% | Test Acc: 96.810%
Epoch  6 | Loss: 0.0536 | Train Acc:  98.35% | Test Acc: 96.270%
Epoch  7 | Loss: 0.0444 | Train Acc:  98.59% | Test Acc: 97.900%
Epoch  8 | Loss: 0.0282 | Train Acc:  99.09% | Test Acc: 97.890%
Epoch  9 | Loss: 0.0155 | Train Acc:  99.47% | Test Acc: 98.200%
Epoch 10 | Loss: 0.0104 | Train Acc:  99.66% | Test Acc: 98.510%
Epoch 11 | Loss: 0.0037 | Train Acc:  99.90% | Test Acc: 98.480%
Epoch 12 | Loss: 0.0009 | Train Acc:  99.98% | Test Acc: 98.480%
Epoch 13 | Loss: 0.0004 | Train Acc: 100.00% | Test Acc: 98.500%
Epoch 14 | Loss: 0.0003 | Train Acc: 100.00% | Test Acc: 98.520%
Epoch 15 | Loss: 0.0003 | Train Acc: 100.00% | Test Acc

#transfer_2ch_closure.py

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from clos import Clos, transfer_fc_to_clos          # Your custom CLOS module
from train_mnist import MNIST_Net, test             # Your trained MLP from previous script

import random
import numpy as np


def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")


    model = MNIST_Net().to(device)
    model.load_state_dict(torch.load("mnist_784_256_10.pth", map_location=device))
    print("Original model loaded (mnist_784_256_10.pth)")
    fc = model.fc1

    transform = transforms.Compose([transforms.ToTensor()])
    test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False,
                             num_workers=2, pin_memory=True)

    accuracy_original = test(model, test_loader, use_amp=True, device=device)
    print("Accuracy of original Linear layer:", accuracy_original)


    best_acc = 0.0
    best_i = -1

    for i in range(10):
        set_all_seeds(42 + i)

        # set_all_seeds(42)

        print(f"\n===== Run {i} (seed = {42 + i}) =====")

        clos = transfer_fc_to_clos(
            fc=fc,
            channel=2,
            max_steps=10000,
            W_lr=3e-3,           # 3e-3   0.1
            B_lr=1e-3,           #  1e-3    0.3
            verbose=True
        )

        model.fc1 = clos
        model.eval()

        accuracy_clos = test(model, test_loader, use_amp=True, device=device)
        print(f"Accuracy after replacing with CLOS {i}th: {accuracy_clos:.3f}%")

        if accuracy_clos > best_acc:
            best_acc = accuracy_clos
            best_i = i
            torch.save(clos.state_dict(), "clos_784_best_test.pth")
            print(f"  → NEW BEST! {best_acc:.3f}% (run {i}) → saved to clos_784_best_test.pth\n")

    print(f"\n===== Туршилт дууслаа =====")
    print(f"Хамгийн сайн үр дүн: {best_acc:.3f}% (run {best_i}, seed={42 + best_i})")


if __name__ == "__main__":
    main()

#clos2chtest.py

In [ ]:
import torch
from clos import Clos
from train_mnist import MNIST_Net
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# ==========================
# 1. Тохиргоо
# ==========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Ашиглаж байгаа төхөөрөмж: {device}")
use_amp = True if device.type == "cuda" else False

# ==========================
# 2. Dataset (MNIST)
# ==========================
model = MNIST_Net()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
model = MNIST_Net().to(device)
model.load_state_dict(torch.load("mnist_784_256_10.pth", map_location=device))
print(f"Нийт linear параметр: {sum(p.numel() for p in model.fc1.parameters()):,}")
clos = Clos(784, 784, channel=2).to(device)
clos.load_state_dict(torch.load("clos_784_best_test.pth", map_location=device))
model.fc1 = clos
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)
model.eval()
print("Clone done:", model)
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f"Тестийн нарийвчлал: {100 * correct / total:.2f}%") # want to increase accuracy to >97%
print(f"Нийт clos параметр: {sum(p.numel() for p in clos.parameters()):,}")

